In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100`% !important; }</style>"))
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

In [2]:
from apiutils import WebIO
from ioutils import FileIO, HTMLIO
from fileutils import FileInfo, DirInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame, concat
from sys import prefix
from listUtils import getFlatList
from musicdb import PanDBIO
mp    = MasterParams(verbose=True)
io    = FileIO()
wio   = WebIO()
hio   = HTMLIO()
mdbpd = MusicDBPermDir()

MasterBasic()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb
MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts', 'Dates']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia', 'AppearanceMedia', 'TechnicalMedia', 'MixMedia', 'BootlegMedia', 'AltMediaMedia', 'OtherMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM', 'Beatport', 'Traxsource', 'MyMixTapez', 'ClassicalArchives', 'JioSaavn']


In [3]:
from lib import classicalarchives
mio   = classicalarchives.MusicDBIO(verbose=True, mkDirs=False)
webio = classicalarchives.RawWebData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

MusicDBBaseDirs(db=ClassicalArchives)
   RawDataDir     = /Volumes/Piggy/Discog/artists-classicalarchives
   ModValDataDir  = /Volumes/Seagate/Discog/artists-classicalarchives-db
   MetaDataDir    = /Volumes/Seagate/Discog/artists-classicalarchives-db/metadata
   SummaryDataDir = /Users/tgadfort/Music/Discog/db-classicalarchives
ClassicalArchives SummaryData
  ==> Basic
  ==> Media
  ==> Genre
  ==> Bio
  ==> Link
  ==> Metric
  ==> Dates
ParseRawDataUtils(mdbdata, mdbdir) [ClassicalArchives]
ClassicalArchives ModValMetaData
  ==> Basic (Universal)
  ==> Media (Media)
  ==> Dates (Universal)
Saving Perminant ClassicalArchives DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/ClassicalArchives


In [4]:
#for modVal in range(78,100):
#    mio.prd.parsePerformerData(modVal, force=True)

In [ ]:
bsdata = hio.get(io.get("/Volumes/Piggy/Discog/artists-classicalarchives/78/performer/1878.p"))
bsdata

In [4]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
localComposers     = MusicDBData(path=permDir, fname="{0}SearchedForLocalComposers".format(db.lower()))
localPerformers    = MusicDBData(path=permDir, fname="{0}SearchedForLocalPerformers".format(db.lower()))
knownArtists       = {} #mio.data.getSummaryNameData()
searchComposers    = mio.data.getSearchComposersData()
searchPerformers   = mio.data.getSearchPerformersData()
errors             = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [5]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Local Composers:           {0}".format(len(localComposers.get())))
print("   Local Performers:          {0}".format(len(localPerformers.get())))
print("   Errors:                    {0}".format(len(errors.get())))
print("   Search Composers:          {0}".format(len(searchComposers)))
print("   Search Performers:         {0}".format(len(searchPerformers)))
print("   Known Summary IDs:         {0}".format(len(knownArtists)))

ClassicalArchives Search Results
   Local Composers:           6190
   Local Performers:          18517
   Errors:                    1
   Search Composers:          7547
   Search Performers:         18518
   Known Summary IDs:         0


In [35]:
import os
os.environ["LC_ALL"] = "en_US.UTF-8"
os.environ["LANG"] = "en_US.UTF-8"
#export LC_ALL=en_US.UTF-8; export LANG=en_US.UTF-8

In [6]:
import osascript
def getScript(url, savename, dtime):
    dscript = '''
tell application "Safari"
activate
set URL of document 1 to "{0}"
delay {2}
set myString to source of document 1
end tell
set newFile to POSIX file "{1}"
open for access newFile with write permission
write myString to newFile as «class utf8»
close access newFile
'''.format(url, savename, dtime)
    
    return dscript

# Download ModVal Data

## Download Via OSA

In [ ]:
#url = f"https://www.classicalarchives.com/performers/{ch}.html"
from string import ascii_lowercase
aTypes = ["composers", "performers"]
for aType in aTypes:
    for ch in ascii_lowercase:
        url = f"https://www.classicalarchives.com/{aType}/{ch}.html"
        savename = f"/Users/tgadfort/Documents/code310/pandb/note/classicalarchives/{aType}_{ch}.html"
        if FileInfo(savename).exists():
            continue
        print(f"{url: <60} ==> {savename}")
        dscript  = getScript(url, savename, dtime=15)
        code,out,err = osascript.run(dscript)

## Parse OSA Data

In [ ]:
from lib.classicalarchives import MusicDBID
mdbid = MusicDBID()
aTypes = ["composers", "performers"]
saveData = {}
for aType in aTypes:
    artistData = {}
    files  = DirInfo("/Users/tgadfort/Documents/code310/pandb/note/classicalarchives").glob(f"{aType}_*.html")
    for ifile in files:
        fData = {}
        bsdata = hio.get(open(ifile, encoding="latin-1").read())
        listingDiv = bsdata.find("div", {"class": "listing"})
        if listingDiv:
            refs = listingDiv.findAll("a")
            fData.update({mdbid.get(ref): {"Name": ref.text, "Ref": ref.get('href')} for ref in refs})
        print(len(fData),'\t',ifile)
        artistData.update(fData)
    artistData = DataFrame(artistData).T
    artistData = artistData[artistData["Ref"].apply(lambda ref: isinstance(ref,str))]
    saveData[aType] = artistData

In [ ]:
mio.data.saveSearchComposersData(data=saveData['composers'])
mio.data.saveSearchPerformersData(data=saveData['performers'])

# Download Composer Data

In [7]:
mio   = classicalarchives.MusicDBIO(verbose=False,local=True,mkDirs=False)

In [10]:
useSearchData = True
if useSearchData is True:
    composerNames      = searchComposers #.sort_values(by="Num", ascending=False)
    localComposersDict = localComposers.get()
    composerNamesToGet = composerNames[~composerNames.index.isin(localComposersDict.keys())].sample(frac=1)

    print("# {0} Search Results".format(db))
    print("#   Available Names:      {0}".format(len(composerNames)))
    print("#   Known Artist Names:   {0}".format(len(localComposersDict)))
    print("#   Artist Names To Get:  {0}".format(len(composerNamesToGet)))

# ClassicalArchives Search Results
#   Available Names:      7547
#   Known Artist Names:   7547
#   Artist Names To Get:  0


In [43]:
if False:
    localComposersDict  = localComposers.get()
    for i,(composerID,row) in enumerate(composerNamesToGet.iterrows()):
        composerName = row["Name"]
        composerRef  = row["Ref"]
        localComposersDict[composerID] = composerName
        if len(localComposersDict) == 500:
            break

    print("Saving {0} {1} Composers Data".format(len(localComposersDict), db))
    localComposers.save(data=localComposersDict)

In [9]:
from timeutils import Timestat, TermTime
import random

ts = Timestat("Getting {0} composerIDs".format(db))
tt = TermTime("tomorrow", "9:50am")
#tt = TermTime("today", "7:00pm")
maxN = 5000000

n  = 0
localComposersDict  = localComposers.get()
searchedForErrors   = errors.get()
N = composerNamesToGet.shape[0]

for i,(composerID,row) in enumerate(composerNamesToGet.iterrows()):
    composerName = row["Name"]
    composerRef  = row["Ref"]
    if localComposersDict.get(composerID) is not None:
        continue
    #if searchedForErrors.get(composerID) is not None:
    #    continue
        
    url = f"https://www.classicalarchives.com{composerRef}"
    savename = f"/Users/tgadfort/Desktop/ClassicalArchives/Composer/{composerID}.html"
    #if FileInfo(savename).exists():
    #    continue
    print(f"{i+1: >5}/{N: <10}{url: <60} ==> ", end="")
    dscript  = getScript(url, savename, dtime=7+random.randint(0,5))
    code,out,err = osascript.run(dscript)
    if FileInfo(savename).exists():
        print(f"{composerID}")
        localComposersDict[composerID] = composerName
    else:
        searchedForErrors[composerID] = composerName
        print(f"Error in download.")
        
    n += 1
        
    if n % 10 == 0 or n >= maxN:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Composers Data".format(len(localComposersDict), db))
        localComposers.save(data=localComposersDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished() or n >= maxN:
            break
    
ts.stop()
print("Saving {0} {1} Composers Data".format(len(localComposersDict), db))
localComposers.save(data=localComposersDict)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

Process [Getting ClassicalArchives composerIDs] Start    ==> Time Is 2022-06-26 23:00:14
========================= termTime(day=tomorrow,time=9:50am) =========================
   ====> Terminate Time Set To 2022-06-27 09:50:00 <====
   ====> Will Terminate Process 10 Hours and 49 Minutes From Now
    1/1357      https://www.classicalarchives.com/composer/83711.html        ==> 83711
    2/1357      https://www.classicalarchives.com/composer/32213.html        ==> 32213
    3/1357      https://www.classicalarchives.com/composer/3210.html         ==> 3210
    4/1357      https://www.classicalarchives.com/composer/67344.html        ==> 67344
    5/1357      https://www.classicalarchives.com/composer/37885.html        ==> 37885
    6/1357      https://www.classicalarchives.com/composer/79284.html        ==> 79284
    7/1357      https://www.classicalarchives.com/composer/43198.html        ==> 43198
    8/1357      https://www.classicalarchives.com/composer/27776.html        ==> 27776
    9/1

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

   11/1357      https://www.classicalarchives.com/composer/73207.html        ==> 73207
   12/1357      https://www.classicalarchives.com/composer/79038.html        ==> 79038
   13/1357      https://www.classicalarchives.com/composer/66163.html        ==> 66163
   14/1357      https://www.classicalarchives.com/composer/76642.html        ==> 76642
   15/1357      https://www.classicalarchives.com/composer/21703.html        ==> 21703
   16/1357      https://www.classicalarchives.com/composer/93454.html        ==> 93454
   17/1357      https://www.classicalarchives.com/composer/21807.html        ==> 21807
   18/1357      https://www.classicalarchives.com/composer/12616.html        ==> 12616
   19/1357      https://www.classicalarchives.com/composer/88768.html        ==> 88768
   20/1357      https://www.classicalarchives.com/composer/69595.html        ==> 69595
20/?       : Process [Getting ClassicalArchives composerIDs] Has Run For 3 Minutes and 19 Seconds.
Saving 6210 ClassicalArchives C

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

   21/1357      https://www.classicalarchives.com/composer/43937.html        ==> 43937
   22/1357      https://www.classicalarchives.com/composer/95640.html        ==> 95640
   23/1357      https://www.classicalarchives.com/composer/50321.html        ==> 50321
   24/1357      https://www.classicalarchives.com/composer/84715.html        ==> 84715
   25/1357      https://www.classicalarchives.com/composer/11789.html        ==> 11789
   26/1357      https://www.classicalarchives.com/composer/3592.html         ==> 3592
   27/1357      https://www.classicalarchives.com/composer/10540.html        ==> 10540
   28/1357      https://www.classicalarchives.com/composer/8649.html         ==> 8649
   29/1357      https://www.classicalarchives.com/composer/70518.html        ==> 70518
   30/1357      https://www.classicalarchives.com/composer/85477.html        ==> 85477
30/?       : Process [Getting ClassicalArchives composerIDs] Has Run For 5 Minutes.
Saving 6220 ClassicalArchives Composers Data


...

 1331/1357      https://www.classicalarchives.com/composer/8055.html         ==> 8055
 1332/1357      https://www.classicalarchives.com/composer/11074.html        ==> 11074
 1333/1357      https://www.classicalarchives.com/composer/37159.html        ==> 37159
 1334/1357      https://www.classicalarchives.com/composer/12172.html        ==> 12172
 1335/1357      https://www.classicalarchives.com/composer/61969.html        ==> 61969
 1336/1357      https://www.classicalarchives.com/composer/36129.html        ==> 36129
 1337/1357      https://www.classicalarchives.com/composer/6457.html         ==> 6457
 1338/1357      https://www.classicalarchives.com/composer/48320.html        ==> 48320
 1339/1357      https://www.classicalarchives.com/composer/52298.html        ==> 52298
 1340/1357      https://www.classicalarchives.com/composer/34307.html        ==> 34307
1340/?     : Process [Getting ClassicalArchives composerIDs] Has Run For 3 Hours and 58 Minutes.
Saving 7530 ClassicalArchives Compo

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

 1341/1357      https://www.classicalarchives.com/composer/85051.html        ==> 85051
 1342/1357      https://www.classicalarchives.com/composer/50434.html        ==> 50434
 1343/1357      https://www.classicalarchives.com/composer/3447.html         ==> 3447
 1344/1357      https://www.classicalarchives.com/composer/10055.html        ==> 10055
 1345/1357      https://www.classicalarchives.com/composer/32659.html        ==> 32659
 1346/1357      https://www.classicalarchives.com/composer/76685.html        ==> 76685
 1347/1357      https://www.classicalarchives.com/composer/88720.html        ==> 88720
 1348/1357      https://www.classicalarchives.com/composer/51622.html        ==> 51622
 1349/1357      https://www.classicalarchives.com/composer/48137.html        ==> 48137
 1350/1357      https://www.classicalarchives.com/composer/13978.html        ==> 13978
1350/?     : Process [Getting ClassicalArchives composerIDs] Has Run For 4 Hours and 18 Seconds.
Saving 7540 ClassicalArchives Comp

Waiting:   0%|          | 0/50 [00:00<?, ?it/s]

 1351/1357      https://www.classicalarchives.com/composer/59579.html        ==> 59579
 1352/1357      https://www.classicalarchives.com/composer/6966.html         ==> 6966
 1353/1357      https://www.classicalarchives.com/composer/14572.html        ==> 14572
 1354/1357      https://www.classicalarchives.com/composer/51207.html        ==> 51207
 1355/1357      https://www.classicalarchives.com/composer/48971.html        ==> 48971
 1356/1357      https://www.classicalarchives.com/composer/60895.html        ==> 60895
 1357/1357      https://www.classicalarchives.com/composer/86607.html        ==> 86607
Process [Getting ClassicalArchives composerIDs] Ran For 4 Hours and 1 Minute    ==> Time Is 2022-06-27 03:01:59
Saving 7547 ClassicalArchives Composers Data
Saving 1 ClassicalArchives Errors


# Download Performer Data

In [11]:
mio   = classicalarchives.MusicDBIO(verbose=False,local=True,mkDirs=False)

In [15]:
useSearchData = True

if useSearchData is True:
    performerNames      = searchPerformers #.sort_values(by="Num", ascending=False)
    localPerformersDict = localPerformers.get()
    performerNamesToGet = performerNames[~performerNames.index.isin(localPerformersDict.keys())].sample(frac=1)

    print("# {0} Search Results".format(db))
    print("#   Available Names:      {0}".format(len(performerNames)))
    print("#   Known Artist Names:   {0}".format(len(localPerformersDict)))
    print("#   Artist Names To Get:  {0}".format(len(performerNamesToGet)))
    
#   Artist Names To Get:  18518

# ClassicalArchives Search Results
#   Available Names:      18518
#   Known Artist Names:   3501
#   Artist Names To Get:  15017


In [16]:
from timeutils import Timestat, TermTime
import random

ts = Timestat("Getting {0} performerIDs".format(db))
tt = TermTime("tomorrow", "9:50am")
#tt = TermTime("today", "10:00pm")
maxN = 50000

n  = 0
localPerformersDict = localPerformers.get()
searchedForErrors   = errors.get()
N = performerNamesToGet.shape[0]

for i,(performerID,row) in enumerate(performerNamesToGet.iterrows()):
    performerName = row["Name"]
    performerRef  = row["Ref"]
    if localPerformersDict.get(performerID) is not None:
        continue
    if searchedForErrors.get(performerID) is not None:
        continue
        
    url = f"https://www.classicalarchives.com{performerRef}"
    savename = f"/Users/tgadfort/Desktop/ClassicalArchives/Performer/{performerID}.html"
    if FileInfo(savename).exists():
        continue
    print(f"{i+1: >5}/{N: <10}{url: <60} ==> ", end="")
    dscript  = getScript(url, savename, dtime=7+random.randint(0,5))
    code,out,err = osascript.run(dscript)
    if FileInfo(savename).exists():
        print(f"{performerID}")
        localPerformersDict[performerID] = performerName
    else:
        searchedForErrors[performerID] = performerName
        print(f"Error in download.")
        
    n += 1
        
    if n % 25 == 0 or n >= maxN:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} performers Data".format(len(localPerformersDict), db))
        localPerformers.save(data=localPerformersDict)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        webio.wait(10.0)
        if tt.isFinished() or n >= maxN:
            break
    
ts.stop()
print("Saving {0} {1} performers Data".format(len(localPerformersDict), db))
localPerformers.save(data=localPerformersDict)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

Process [Getting ClassicalArchives performerIDs] Start    ==> Time Is 2022-06-28 22:54:56
========================= termTime(day=tomorrow,time=9:50am) =========================
   ====> Terminate Time Set To 2022-06-29 09:50:00 <====
   ====> Will Terminate Process 10 Hours and 55 Minutes From Now
    1/15017     https://www.classicalarchives.com/artist/69619.html          ==> 69619
    2/15017     https://www.classicalarchives.com/artist/30081.html          ==> 30081
    3/15017     https://www.classicalarchives.com/artist/64175.html          ==> 64175
    4/15017     https://www.classicalarchives.com/artist/117057.html         ==> 117057
    5/15017     https://www.classicalarchives.com/artist/87083.html          ==> 87083
    6/15017     https://www.classicalarchives.com/artist/42871.html          ==> 42871
    7/15017     https://www.classicalarchives.com/artist/38021.html          ==> 38021
    8/15017     https://www.classicalarchives.com/artist/29722.html          ==> 29722
    

Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

   26/15017     https://www.classicalarchives.com/ensemble/8110.html         ==> 8110
   27/15017     https://www.classicalarchives.com/artist/27331.html          ==> 27331
   28/15017     https://www.classicalarchives.com/artist/15563.html          ==> 15563
   29/15017     https://www.classicalarchives.com/artist/19271.html          ==> 19271
   30/15017     https://www.classicalarchives.com/artist/83853.html          ==> 83853
   31/15017     https://www.classicalarchives.com/artist/14706.html          ==> 14706
   32/15017     https://www.classicalarchives.com/artist/38343.html          ==> 38343
   33/15017     https://www.classicalarchives.com/artist/29932.html          ==> 29932
   34/15017     https://www.classicalarchives.com/artist/48241.html          ==> 48241
   35/15017     https://www.classicalarchives.com/artist/40426.html          ==> 40426
   36/15017     https://www.classicalarchives.com/artist/15804.html          ==> 15804
   37/15017     https://www.classicalarchive

Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

   51/15017     https://www.classicalarchives.com/artist/86584.html          ==> 86584
   52/15017     https://www.classicalarchives.com/artist/79291.html          ==> 79291
   53/15017     https://www.classicalarchives.com/ensemble/13226.html        ==> 13226
   54/15017     https://www.classicalarchives.com/artist/67333.html          ==> 67333
   55/15017     https://www.classicalarchives.com/artist/59865.html          ==> 59865
   56/15017     https://www.classicalarchives.com/artist/73723.html          ==> 73723
   57/15017     https://www.classicalarchives.com/artist/63111.html          ==> 63111
   58/15017     https://www.classicalarchives.com/artist/71985.html          ==> 71985
   59/15017     https://www.classicalarchives.com/artist/59555.html          ==> 59555
   60/15017     https://www.classicalarchives.com/ensemble/22123.html        ==> 22123
   61/15017     https://www.classicalarchives.com/artist/99700.html          ==> 99700
   62/15017     https://www.classicalarchiv

...

 2376/15017     https://www.classicalarchives.com/ensemble/9229.html         ==> 9229
 2377/15017     https://www.classicalarchives.com/artist/3664.html           ==> 3664
 2378/15017     https://www.classicalarchives.com/artist/57172.html          ==> 57172
 2379/15017     https://www.classicalarchives.com/ensemble/3052.html         ==> 3052
 2380/15017     https://www.classicalarchives.com/ensemble/18324.html        ==> 18324
 2381/15017     https://www.classicalarchives.com/artist/62075.html          ==> 62075
 2382/15017     https://www.classicalarchives.com/artist/52316.html          ==> 52316
 2383/15017     https://www.classicalarchives.com/artist/78735.html          ==> 78735
 2384/15017     https://www.classicalarchives.com/artist/72226.html          ==> 72226
 2385/15017     https://www.classicalarchives.com/artist/30642.html          ==> 30642
 2386/15017     https://www.classicalarchives.com/artist/69678.html          ==> 69678
 2387/15017     https://www.classicalarchives.

Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

 2401/15017     https://www.classicalarchives.com/artist/62993.html          ==> 62993
 2402/15017     https://www.classicalarchives.com/artist/73982.html          ==> 73982
 2403/15017     https://www.classicalarchives.com/artist/39804.html          ==> 39804
 2404/15017     https://www.classicalarchives.com/artist/6887.html           ==> 6887
 2405/15017     https://www.classicalarchives.com/artist/16597.html          ==> 16597
 2406/15017     https://www.classicalarchives.com/ensemble/6836.html         ==> 6836
 2407/15017     https://www.classicalarchives.com/artist/77122.html          ==> 77122
 2408/15017     https://www.classicalarchives.com/artist/65947.html          ==> 65947
 2409/15017     https://www.classicalarchives.com/artist/106014.html         ==> 106014
 2410/15017     https://www.classicalarchives.com/ensemble/18386.html        ==> 18386
 2411/15017     https://www.classicalarchives.com/artist/9261.html           ==> 9261
 2412/15017     https://www.classicalarchives

Waiting:   0%|          | 0/100 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
localPerformers.save(data=localPerformersDict)

In [ ]:
from lib.classicalarchives import moveLocalFiles, removeLocalFiles
#moveLocalFiles()
removeLocalFiles()
#localPerformers.save(data=localPerformersDict)

In [ ]:
mio.prd.parseComposerData(modVal=1, force=True)
mio.prd.mergeModValData(modVal=1)

# Download & Parse

In [21]:
from fileutils import DirInfo
from ioutils import FileIO
aTypeDir = "Composer"
mioLocal  = DirInfo(f"/Users/tgadfort/Desktop/ClassicalArchives/{aTypeDir}")
io        = FileIO()
print("  ==> Finding Files in {0}: ".format(mioLocal.str), end="")
files = list(mioLocal.glob("*.htm*"))
print("  ==> Found {0} Files".format(len(files)))

  ==> Finding Files in /Users/tgadfort/Desktop/ClassicalArchives/Composer: Running glob(/Users/tgadfort/Desktop/ClassicalArchives/Composer/*.htm*)
  ==> Found 1283 Files


In [40]:
composerID="23701"
composerRef=f"/composer/{composerID}.html"
url = f"https://www.classicalarchives.com{composerRef}"
savename = f"/Users/tgadfort/Desktop/ClassicalArchives/Composer/{composerID}.html"
print(f"{url: <60} ==> ", end="")
dscript  = getScript(url, savename, dtime=3+random.randint(0,5))
code,out,err = osascript.run(dscript)
print(f"{savename}")

https://www.classicalarchives.com/composer/23701.html        ==> /Users/tgadfort/Desktop/ClassicalArchives/Composer/23701.html


In [24]:
names = mio.data.getSummaryNameData()

In [29]:
names[names.index.isin([FileInfo(ifile).basename for ifile in files])].head(30)

36100                              Ron Mazurek
25500                       George David Weiss
61400                           Stefan Safsten
95700                              Simon Wawer
47900                            Edmund Meisel
8700                     Peter Dodds McCormick
87700                             Tom Dempster
3500                          Sergei Vasilenko
63401                           Vladimir Genin
43401                    Raffaele D'Alessandro
56301                            David Behrman
23701                           Carlos Farias
86201                      Wilhelm Baumgartner
42501                    Sigismund von Neukomm
92101                             Charles Lamb
64501                          Alice Hawthorne
43201                              Gerald Near
31001                        Salvatore Messina
34901    Claude - Herbert Schonberg - Kretzmer
33001                      Josef Bardanashvili
36501                      Niccol Castiglioni
96902        

In [33]:
ifile = '/Users/tgadfort/Desktop/ClassicalArchives/Composer/23701.html'
data  = open(ifile, encoding="ascii").read()

UnicodeDecodeError: 'ascii' codec can't decode byte 0x96 in position 286: ordinal not in range(128)

In [32]:
data

'\n\n\n\n\n<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">\n<html xmlns="http://www.w3.org/1999/xhtml">\n    <head>\n        <meta http-equiv="Content-Type" content="text/html; charset=utf-8" />\n        <title>Carlos Fari\x96as - Classical Archives</title>\n<meta http-equiv="PICS-Label" content=\'(PICS-1.1 "http://www.classify.org/safesurf/" l gen true for "https://www.classicalarchives.com/" r (SS~~000 1))\'/>\n<meta http-equiv="pics-label" content=\'(pics-1.1 "http://www.icra.org/ratingsv02.html" comment "Single file v2.0" l gen true for "https://www.classicalarchives.com" r (nz 1 vz 1 lz 1 oz 1 cb 1) "http://www.rsac.org/ratingsv01.html" l gen true for "https://www.classicalarchives.com" r (n 0 s 0 v 0 l 0))\'/>\n<meta http-equiv="content-language" content="en"/>\n<meta name="Robots" content="INDEX, FOLLOW"/>\n<meta name="Author" content="Classical Archives LLC"/>\n<meta name="Subject" content="Classical Musi

In [ ]:

        files = list(mioLocal.glob("*.htm*"))
        ts = Timestat("Moving {0} Local Files To Global Directories".format(len(files)))
        for n,ifile in enumerate(files):
            if (n+1) % 25 == 0:
                ts.update(n=n+1,N=len(files))
            dbID    = FileInfo(ifile).basename
            modVal  = mioGlobal.getModVal(dbID)
            dstFile = FileInfo(eval(f"mioGlobal.data.getRaw{aTypeDir}Filename(modVal,dbID)"))

In [ ]:
from utils import PoolIO
pio = PoolIO("ClassicalArchives")
#pio.parse(force=True)
pio.merge()
pio.meta()
pio.sum()
pio.search()

In [ ]:
# Multiple pages
#https://www.classicalarchives.com/artist/6138.html

In [ ]:
from collections import Counter
cntr = Counter()
for modVal in range(100):
    data = mio.data.getModValData(modVal)
    for k,v in data.iteritems():
        for name,vals in v.mediaCounts.counts.items():
            cntr[name] += vals

In [ ]:
cntr.most_common()

In [ ]:
from lib.classicalarchives import RawDBData
rdbData = RawDBData(debug=False)
retval = rdbData.getPerformerData('/Volumes/Piggy/Discog/artists-classicalarchives/4/performer/105604.p')

In [ ]:
retval.show()

In [ ]:
retval.media.media["Performances"][0].get()

In [ ]:
bsdata = hio.get(io.get('/Volumes/Piggy/Discog/artists-classicalarchives/4/performer/105604.p'))

In [ ]:
bsdata

In [ ]:
jsonData['albums']

In [ ]:
jsonLines = [line.strip().split(" = ")[-1] for line in jsData.split("\n")]

In [ ]:
jsonLines

In [ ]:

try:
    jsonData = [json.loads(jsonLine[:-3]) for jsonLine in jsonLines]
except:
    jsonData = []